In [2]:
import os
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import URL, create_engine, text

# CONFIGURATION

In [3]:
PROJECT_ROOT = Path.cwd()

if not (PROJECT_ROOT / ".env").exists():
    PROJECT_ROOT = Path.cwd().parent.parent

load_dotenv(PROJECT_ROOT / ".env")

CSV_FILES = [
    PROJECT_ROOT / "mysql" / "data" / "sales.csv",
    PROJECT_ROOT / "mysql" / "data" / "sales_2.csv",
]

print("Project root:", PROJECT_ROOT)
print("CSV files found:", [file.exists() for file in CSV_FILES])

Project root: /media/harsh/NewVolume/ETL_Pipeline
CSV files found: [True, True]


In [5]:
MYSQL_USER = os.getenv("DB_USER")
MYSQL_PASSWORD = os.getenv("DB_PASSWORD")
MYSQL_HOST = os.getenv("DB_HOST", "localhost")
MYSQL_PORT = int(os.getenv("DB_PORT", "3306"))
MYSQL_DATABASE = os.getenv("DB_NAME", "sales_etl")

TARGET_TABLE = "sales_orders"

assert MYSQL_USER, "DB_USER is missing from .env"
assert MYSQL_PASSWORD, "DB_PASSWORD is missing from .env"

print("Database configuration loaded.")
print("Database:", MYSQL_DATABASE)
print("Target table:", TARGET_TABLE)

Database configuration loaded.
Database: sales_etl
Target table: sales_orders


## Database Connection

In [6]:
db_url = URL.create(
    drivername="mysql+pymysql",
    username=MYSQL_USER,
    password=MYSQL_PASSWORD,
    host=MYSQL_HOST,
    port=MYSQL_PORT,
    database=MYSQL_DATABASE,
)

engine = create_engine(db_url)

with engine.connect() as connection:
    print("Connected successfully to:", connection.execute(
        text("SELECT DATABASE()")
    ).scalar())

Connected successfully to: sales_etl


### STEP 1: READ CSV FILE (Extract)

In [7]:
# STEP 1: EXTRACT — Read both CSV files

dataframes = []

for file in CSV_FILES:
    df = pd.read_csv(file)
    print(f"Loaded {len(df)} rows from {file.name}")
    dataframes.append(df)

raw_df = pd.concat(dataframes, ignore_index=True)

print("\nTotal rows extracted:", len(raw_df))
display(raw_df.head())

Loaded 10 rows from sales.csv
Loaded 10 rows from sales_2.csv

Total rows extracted: 20


,OrderId,Product,Category,SalesAmount,OrderDate,Region,CustomerName
0,1001,Laptop Pro 13,Electronics,1000.0,2025-01-05,West,Alice Johnson
1,1002,Mechanical Keyboard,Electronics,95.5,2025-01-06,East,Bob Smith
2,1003,Desk Chair Ergonomic,Furniture,350.0,2025-01-06,West,Charlie Brown
3,1004,Coffee Maker Deluxe,Home Goods,150.0,2025-01-07,Central,Diana Prince
4,1005,Wireless Mouse,Electronics,25.0,2025-01-07,East,Evan Peters


## STEP 2: TRANSFORMATIONS (Transform)

In [8]:
# STEP 2: TRANSFORM — Clean and validate the data

df = raw_df.copy()

# Remove duplicate order IDs
df = df.drop_duplicates(subset=["OrderId"], keep="first")

# Convert columns to appropriate data types
df["OrderId"] = pd.to_numeric(df["OrderId"], errors="coerce")
df["SalesAmount"] = pd.to_numeric(df["SalesAmount"], errors="coerce")
df["OrderDate"] = pd.to_datetime(df["OrderDate"], errors="coerce")

# Remove rows with missing required values
required_columns = [
    "OrderId",
    "Product",
    "Category",
    "SalesAmount",
    "OrderDate",
    "Region",
    "CustomerName",
]

df = df.dropna(subset=required_columns)

# Reject invalid order IDs and negative sales amounts
df = df[
    (df["OrderId"] > 0)
    & (df["OrderId"] % 1 == 0)
    & (df["SalesAmount"] >= 0)
].copy()

# Convert database columns to the correct types
df["OrderId"] = df["OrderId"].astype(int)
df["OrderDate"] = df["OrderDate"].dt.date

print("Rows extracted:", len(raw_df))
print("Rows after cleaning:", len(df))
print("Rows rejected:", len(raw_df) - len(df))

display(df.head())

Rows extracted: 20
Rows after cleaning: 19
Rows rejected: 1


,OrderId,Product,Category,SalesAmount,OrderDate,Region,CustomerName
0,1001,Laptop Pro 13,Electronics,1000.0,2025-01-05,West,Alice Johnson
1,1002,Mechanical Keyboard,Electronics,95.5,2025-01-06,East,Bob Smith
2,1003,Desk Chair Ergonomic,Furniture,350.0,2025-01-06,West,Charlie Brown
3,1004,Coffee Maker Deluxe,Home Goods,150.0,2025-01-07,Central,Diana Prince
4,1005,Wireless Mouse,Electronics,25.0,2025-01-07,East,Evan Peters


## STEP 3: LOAD INTO MYSQL (Load)

In [12]:


# STEP 3: LOAD — Insert or update cleaned records in MySQL

from sqlalchemy.dialects.mysql import insert

columns = [
    "OrderId",
    "Product",
    "Category",
    "SalesAmount",
    "OrderDate",
    "Region",
    "CustomerName",
]

load_df = df[columns].copy()

# Insert new orders; update matching OrderIds
# if those orders already exist.
table = __import__("sqlalchemy").Table(
    TARGET_TABLE,
    __import__("sqlalchemy").MetaData(),
    autoload_with=engine,
)

with engine.begin() as connection:
    for start in range(0, len(load_df), 100):
        batch = load_df.iloc[start:start + 100].to_dict("records")

        statement = insert(table).values(batch)

        update_values = {
            column: statement.inserted[column]
            for column in columns
            if column != "OrderId"
        }

        statement = statement.on_duplicate_key_update(**update_values)
        connection.execute(statement)

print(f"ETL load completed for {len(load_df)} valid records.")



ETL load completed for 19 valid records.


# STEP 4: VERIFY — Check records in MySQL

In [13]:
with engine.connect() as connection:
    total_rows = connection.execute(
        text("SELECT COUNT(*) FROM sales_orders")
    ).scalar()

    rejected_orders = connection.execute(
        text("""
            SELECT OrderId, Product, SalesAmount
            FROM sales_orders
            WHERE SalesAmount < 0
        """)
    ).fetchall()

    print("Total rows in MySQL:", total_rows)
    print("Invalid negative-amount records in MySQL:", len(rejected_orders))

    print("\nSample loaded records:")
    result = connection.execute(
        text("""
            SELECT OrderId, Product, SalesAmount, Region
            FROM sales_orders
            ORDER BY OrderId
            LIMIT 5
        """)
    )

    for row in result:
        print(row)


Total rows in MySQL: 19
Invalid negative-amount records in MySQL: 0

Sample loaded records:
(1001, 'Laptop Pro 13', Decimal('1000.00'), 'West')
(1002, 'Mechanical Keyboard', Decimal('95.50'), 'East')
(1003, 'Desk Chair Ergonomic', Decimal('350.00'), 'West')
(1004, 'Coffee Maker Deluxe', Decimal('150.00'), 'Central')
(1005, 'Wireless Mouse', Decimal('25.00'), 'East')


# STEP 5 - TESTING

In [14]:

with engine.connect() as connection:
    count = connection.execute(
        text("SELECT COUNT(*) FROM sales_orders")
    ).scalar()

print("Verified destination row count:", count)
assert count == 19, "Unexpected number of rows in the destination table."
print("Validation passed!")


Verified destination row count: 19
Validation passed!
